# Gestire dati mancanti
L'operare su dataset in cui alcuni valori sono mancanti è un problema tipico del data preprocessing.
Vediamo i metodi da applicare in questo caso, cominciamo importando le librerie che utilizzeremo.

In [5]:
import pandas as pd
import numpy as np

## Creiamo il nostro dataset con valori mancanti
Per i nostri esempi utilizzeremo l'Iris Dataset, questo famoso dataset non presenta alcun valore mancante, quindi creiamone qualcuno noi.<br>
Carichiamo il dataset in un DataFrame.

In [16]:
import pandas as pd
import requests
from io import StringIO

# Disabilita il controllo SSL per la richiesta HTTP
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
response = requests.get(url, verify=False)

# Carica i dati in pandas usando StringIO per simulare un file
data = StringIO(response.text)
iris = pd.read_csv(data, names=["sepal_length", "sepal_width", "petal_length", "petal_width", "class"])

# Visualizzare le prime righe del dataframe
print(iris.head())




   sepal_length  sepal_width  petal_length  petal_width        class
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa


c:\Users\G.Zaffinelli-cons\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'archive.ics.uci.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [17]:
iris_nan = iris.copy()
max_val = iris.shape[0]

samples = np.random.randint(max_val, size=(10)) #Creiamo un vettore di 10 numeri casuali tra 0 e il numero di osservazioni
iris_nan.loc[samples,'petal_length']=None #Sostituiamo il valore di "petal_length" per ognuna delle 10 osservazioni con un valore non valido

nan_count = iris_nan['petal_length'].isnull().sum() #contiamo il numero di valori non validi all'interno della colonna "petal_legnth"
print("Il dataset ha "+str(nan_count)+" valori mancanti")
iris_nan.to_csv("data/iris_with_nan.csv") # salviamo il dataset così creato all'interno di un file CSV

Il dataset ha 9 valori mancanti


Utilizziamo il DataFrame per caricare il dataset anche in un array numpy

In [18]:
Y = iris_nan["class"].values
X = iris_nan.drop("class",axis=1).values

## - Metodo 1: Rimuovere proprietà o esempi con valori mancanti

Una soluzione drastica consiste nel rimuove gli esempi che presentano valori mancanti utilizzando il metodo dropna.

In [19]:
samples_before = iris_nan.shape[0]
iris_drop = iris_nan.dropna()

samples_after = iris_drop.shape[0]

print("Numero di esempi prima: "+str(samples_before))
print("Numero di esempi dopo: "+str(samples_after))

Numero di esempi prima: 150
Numero di esempi dopo: 141


Se i valori mancanti corrispondono ad un unica feature e questi sono in un numero tale da invalidare l'utilità della feature, allora possiamo semplicemente rimuovere la feature dal nostro DataFrame.

In [20]:
iris_cleaned = iris_nan.dropna(axis=1)
iris_cleaned.columns

Index(['sepal_length', 'sepal_width', 'petal_width', 'class'], dtype='object')

Rinunciare a dati preziosi non è mai una buona cosa, quindi questi metodi vanno evitati ad eccezione di casi estremi, ovvero quando la maggior parte dei valori per una feature o per un esempio sono mancanti.

## - Metodo 2: Imputazione dei dati mancanti

L'imputazione dei dati mancanti consiste nel sostituire i valori con una stima.<br>
Il metodo più comune è **l'imputazione con media** (mean imputation) in cui i valori mancanti vengono sostituiti con il valore medio della proprietà, altri metodi sono l'imputazione con la mediana o con valore più frequente (moda).

### Pandas
Con Pandas possiamo utilizzare il metodo fillna per sostituire i valori mancanti con le stime.

In [21]:
replace_with = iris_nan['petal_length'].mean() # imputazione con media
#replace_with = iris_nan['petal_length'].median() # imputazione con mediana
#replace_with = iris_nan['petal_length'].mode() # imputazione con moda
iris_nan['petal_length'].fillna(replace_with,inplace=True)
nan_count = iris_nan['petal_length'].isnull().sum() #verifichiamo che la colonna "petal_length" non contenga più valori non validi.
print("Il dataset ha "+str(nan_count)+" valori mancanti")

Il dataset ha 0 valori mancanti


C:\Users\G.Zaffinelli-cons\AppData\Local\Temp\ipykernel_5764\269421911.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  iris_nan['petal_length'].fillna(replace_with,inplace=True)


### Numpy e Scikit-learn
Per eseguire l'imputazione di un array numpy possiamo utilizzare la classe Imputer di scikit-learn, il tipo di imputazione può essere specificata nella strategia (mean, median, most_frequent)

In [13]:
nan_count = np.count_nonzero(np.isnan(X))
print("Il dataset ha "+str(nan_count)+" valori mancanti")

Il dataset ha 10 valori mancanti


In [ ]:
from sklearn.preprocessing import Imputer

imp = Imputer(missing_values="NaN", strategy="mean", axis=0) 
X_imp = imp.fit_transform(X)
nan_count = np.count_nonzero(np.isnan(X_imp))
print("Il dataset ha "+str(nan_count)+" valori mancanti")

Il dataset ha 2 valori mancanti prima dell'imputazione


TypeError: SimpleImputer.__init__() got an unexpected keyword argument 'axis'

Dalla versione 0.20 di Scikit-learn la classe Imputer è stata deprecata in favore della classe SimpleImputer. L'utilizzo di questa nuova classe è il medesimo, l'unica differenza sta nel fatto che non accetta come valore del parametro *missing_values* una stringa, piuttosto dobbiamo passargli la costante *nan* di Numpy

In [12]:
import numpy as np
from sklearn.impute import SimpleImputer

imp = SimpleImputer(missing_values = np.nan, strategy = 'mean')
X_imp = imp.fit_transform(X)

nan_count = np.count_nonzero(np.isnan(X_imp))
print("Il dataset ha "+str(nan_count)+" valori mancanti")

Il dataset ha 0 valori mancanti
